In [18]:
!pip install openai-whisper transformers rouge-score webrtcvad librosa soundfile


In [19]:
import whisper
import webrtcvad
import librosa
import numpy as np
from transformers import pipeline
from rouge_score import rouge_scorer


In [20]:
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print("Uploaded:", audio_path)


Saving ya.mp3 to ya (1).mp3
Uploaded: ya (1).mp3


In [21]:
model = whisper.load_model("small")
result = model.transcribe(audio_path)
transcript = result["text"]

print("TRANSCRIPT:\n")
print(transcript)

with open("transcript_m3.txt", "w") as f:
    f.write(transcript)


TRANSCRIPT:

 Real self-confidence doesn't come from shouting affirmations in the mirror. Real self-confidence comes from giving the world irrefutable proof you are who you say you are. I fucking love that.


In [22]:
vad = webrtcvad.Vad(2)

audio, sr = librosa.load(audio_path, sr=16000)
audio_int16 = (audio * 32768).astype(np.int16)

frame_ms = 30
frame_len = int(sr * frame_ms / 1000)

speech_segments = []
current_start = None

# Collect VAD speech segments
for i in range(0, len(audio_int16), frame_len):
    frame = audio_int16[i:i+frame_len].tobytes()
    if len(frame) < frame_len * 2:
        continue
    if vad.is_speech(frame, sr):
        if current_start is None:
            current_start = i / sr
    else:
        if current_start is not None:
            speech_segments.append((current_start, i / sr))
            current_start = None

# Whisper segments
whisper_segments = result["segments"]

# Assign alternating speakers
speaker = 1
diarized_lines = []

for seg in whisper_segments:
    diarized_lines.append(f"[Speaker {speaker}] {seg['text'].strip()}")
    speaker = 2 if speaker == 1 else 1

diarized_text = "\n".join(diarized_lines)

print("DIARIZED TEXT:\n")
print(diarized_text)

with open("diarized_m3.txt", "w") as f:
    f.write(diarized_text)


DIARIZED TEXT:

[Speaker 1] Real self-confidence doesn't come from shouting affirmations in the mirror.
[Speaker 2] Real self-confidence comes from giving the world irrefutable proof you are who you say you are.
[Speaker 1] I fucking love that.


In [24]:
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [25]:
def chunk_text(text, max_chars=2000):
    chunks = []
    while len(text) > max_chars:
        split_point = text[:max_chars].rfind(".")  # split at last period
        if split_point == -1:
            split_point = max_chars
        chunks.append(text[:split_point])
        text = text[split_point:]
    chunks.append(text)
    return chunks

# Chunk diarized text
chunks = chunk_text(diarized_text, max_chars=1800)

summaries = []
for chunk in chunks:
    summary_chunk = summarizer(chunk, max_length=200, min_length=60)[0]["summary_text"]
    summaries.append(summary_chunk)

# Combine partial summaries
summary = " ".join(summaries)

print("\nFINAL SUMMARY:\n")
print(summary)

with open("summary_m3.txt", "w") as f:
    f.write(summary)



Your max_length is set to 200, but your input_length is only 60. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=30)



FINAL SUMMARY:

Real self-confidence doesn't come from shouting affirmations in the mirror. Real self- confidence comes from giving the world irrefutable proof you are who you say you are. [Speaker 1] I fucking love that. I love that you love that I'm a man of my word.


In [26]:
def chunk_text(text, max_chars=1800):
    chunks = []
    while len(text) > max_chars:
        split_point = text[:max_chars].rfind(".")
        if split_point == -1:
            split_point = max_chars
        chunks.append(text[:split_point])
        text = text[split_point:]
    chunks.append(text)
    return chunks

chunks = chunk_text(diarized_text)

summaries = []
for chunk in chunks:
    out = summarizer(chunk, max_length=200, min_length=50)[0]['summary_text']
    summaries.append(out)

summary = " ".join(summaries)

print("\nFINAL SUMMARY:\n")
print(summary)

with open("summary_m3.txt", "w") as f:
    f.write(summary)


Your max_length is set to 200, but your input_length is only 60. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=30)



FINAL SUMMARY:

Real self-confidence doesn't come from shouting affirmations in the mirror. Real self- confidence comes from giving the world irrefutable proof you are who you say you are. [Speaker 1] I fucking love that.


In [27]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
scores = scorer.score(transcript, summary)

print("ROUGE SCORES:\n", scores)

with open("rouge_m3.txt", "w") as f:
    f.write(str(scores))


ROUGE SCORES:
 {'rouge1': Score(precision=0.9428571428571428, recall=1.0, fmeasure=0.9705882352941176), 'rouge2': Score(precision=0.9117647058823529, recall=0.96875, fmeasure=0.9393939393939394), 'rougeL': Score(precision=0.9428571428571428, recall=1.0, fmeasure=0.9705882352941176)}
